**Author:** Li You

**Description:**  
This script automatically crawls song lyrics from [AZLyrics](https://www.azlyrics.com/) in batches and saves
progress files to support resume-safe execution and downstream analysis.

**AI Disclosure:**  
AI tools (e.g., ChatGPT) were used for debugging and documentation support.
All final implementation decisions, testing, and verification were independently completed by the author.

In [ ]:
pip install git+https://github.com/elmoiv/azapi.git

In [ ]:
# Take a test
import re
import requests
from bs4 import BeautifulSoup

def fetch_azlyrics(url: str) -> str:
    """
    Fetch lyrics text from an AZLyrics webpage.

    Inputs:
        url (str): The URL of the AZLyrics page containing the song lyrics.

    Outputs:
        str: Cleaned lyrics text extracted from the page.

    Raises:
        RuntimeError: If the lyrics block cannot be located on the page.
        requests.HTTPError: If the HTTP request fails.
    """

    # Define request headers to simulate a normal browser request
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0 Safari/537.36"
    }

    # Send HTTP request and parse HTML
    r = requests.get(url, headers=headers, timeout=20)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    # AZLyrics typically stores lyrics in a div within the main text container
    main = soup.select_one("div.col-xs-12.col-lg-8.text-center")
    divs = main.find_all("div") if main else soup.find_all("div")

    lyrics = None

    # Search for the div that likely contains the lyrics
    for d in divs:
        if d.get("class") is None and d.get("id") is None:
            text = d.get_text("\n", strip=True)

            # Use simple heuristics to identify the lyrics block
            if len(text.splitlines()) >= 5 and len(text) > 200:
                lyrics = text
                break

    # If no lyrics block was found, raise an error
    if not lyrics:
        raise RuntimeError("Failed to locate lyrics div (page structure may have changed).")

    # Clean excessive blank lines
    lyrics = re.sub(r"\n{3,}", "\n\n", lyrics).strip()

    return lyrics


# Test example
url = "https://www.azlyrics.com/lyrics/taylorswift/blankspace.html"
lyrics = fetch_azlyrics(url)

# Print the first 800 characters of the lyrics
print(lyrics[:800])


Nice to meet you, where you been?
I could show you incredible things
Magic, madness, heaven, sin
Saw you there, and I thought
"Oh my God, look at that face
You look like my next mistake
Love's a game, wanna play?"
Ayy
New money, suit and tie
I can read you like a magazine
Ain't it funny? Rumors fly
And I know you heard about me
So hey, let's be friends
I'm dying to see how this one ends
Grab your passport and my hand
I can make the bad guys good for a weekend
So it's gonna be forever
Or it's gonna go down in flames
You can tell me when it's over, mm
If the high was worth the pain
Got a long list of ex-lovers
They'll tell you I'm insane
'Cause you know I love the players
And you love the game
'Cause we're young and we're reckless
We'll take this way too far
It'll leave you breathless, hmm
O


In [ ]:
import os
import re
import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup


# Pool of realistic browser User-Agent strings to reduce blocking risk
UA_POOL = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0 Safari/537.36",
]


def make_headers():
    """
    Generate HTTP request headers with a randomized (but realistic) User-Agent.

    Inputs:
        None

    Outputs:
        dict: HTTP headers used for web requests to AZLyrics.
    """
    return {
        "User-Agent": random.choice(UA_POOL) + " (Academic research; contact: liyou@uchicago.edu)",
        "Accept-Language": "en-US,en;q=0.9",
    }


# ============================================================
# 1. Fetch lyrics from AZLyrics
# ============================================================

def fetch_azlyrics(url: str, headers: dict, timeout: int = 20) -> str:
    """
    Download an AZLyrics webpage and extract the lyrics text.

    Inputs:
        url (str): The AZLyrics page URL.
        headers (dict): HTTP headers used in the request.
        timeout (int): Maximum request time in seconds.

    Outputs:
        str: Cleaned lyrics text extracted from the page.

    Raises:
        RuntimeError: If the request is blocked or the lyrics block cannot be located.
        requests.HTTPError: If the HTTP request fails.
    """

    # Send request to AZLyrics
    r = requests.get(url, headers=headers, timeout=timeout)

    # Detect potential blocking by the website
    if r.status_code in (403, 429):
        raise RuntimeError(f"Blocked by AZLyrics (HTTP {r.status_code})")

    r.raise_for_status()

    # Parse HTML content
    soup = BeautifulSoup(r.text, "html.parser")

    # AZLyrics stores lyrics in a <div> without class or id
    main = soup.select_one("div.col-xs-12.col-lg-8.text-center")
    divs = main.find_all("div") if main else soup.find_all("div")

    lyrics = None

    # Identify the lyrics block using simple heuristics
    for d in divs:
        if d.get("class") is None and d.get("id") is None:
            text = d.get_text("\n", strip=True)

            # Lyrics blocks tend to be relatively long
            if len(text.splitlines()) >= 5 and len(text) > 200:
                lyrics = text
                break

    # Raise error if lyrics were not found
    if not lyrics:
        raise RuntimeError("Failed to locate lyrics div")

    # Clean excessive blank lines
    lyrics = re.sub(r"\n{3,}", "\n\n", lyrics).strip()

    return lyrics


# ============================================================
# 2. URL construction
# ============================================================

def extract_main_artist(artist: str) -> str:
    """
    Extract the primary artist name when multiple artists are listed.

    Inputs:
        artist (str): Artist string from the dataset.

    Outputs:
        str: The first artist name in the list.
    """
    return artist.split(",")[0].strip() if artist else ""


def normalize_simple(s: str) -> str:
    """
    Normalize text for AZLyrics URL formatting.

    Inputs:
        s (str): Artist or song title.

    Outputs:
        str: Lowercased string with non-alphanumeric characters removed.
    """
    s = s.lower()
    return re.sub(r"[^a-z0-9]", "", s)


def build_azlyrics_url(artist: str, title: str) -> str:
    """
    Construct a standard AZLyrics URL from artist and song title.

    Inputs:
        artist (str): Artist name.
        title (str): Song title.

    Outputs:
        str: AZLyrics page URL for the specified song.
    """

    # Extract the main artist if multiple artists are listed
    artist_main = extract_main_artist(artist)

    # Build the standardized AZLyrics URL
    return (
        "https://www.azlyrics.com/lyrics/"
        f"{normalize_simple(artist_main)}/"
        f"{normalize_simple(title)}.html"
    )

c:\Users\16172\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [ ]:
BATCH_SIZE  = 50
MAX_BATCHES = 2   # Number of batches to run automatically per execution


# Update the file paths below

INPUT_CSV = r"C:\Users\16172\OneDrive\桌面\RA\Music&Finance\uncovered.csv"
OUT_DIR   = r"C:\Users\16172\OneDrive\桌面\RA\Music&Finance\lyrics_batches"

os.makedirs(OUT_DIR, exist_ok=True)

PROGRESS_CSV = os.path.join(OUT_DIR, "covered_with_lyrics_progress.csv")

# ============================================================
# Load data (resume-safe)
# ============================================================
# If a progress file already exists, continue from it;
# otherwise, start from the original input file

if os.path.exists(PROGRESS_CSV):
    df = pd.read_csv(PROGRESS_CSV, encoding="utf-8-sig")
    print("Loaded progress:", PROGRESS_CSV)
else:
    df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")
    print("Loaded input:", INPUT_CSV)

for col in ["azlyrics_url", "lyrics", "lyrics_status"]:
    if col not in df.columns:
        df[col] = ""

# ============================================================
# Automatic multi-batch crawling
# ============================================================

# Track how many batches have been processed in this execution
batch_counter = 0

while True:
    # Identify rows that have not yet been processed
    status = df["lyrics_status"].fillna("").astype(str).str.strip()
    todo_idx = df.index[status == ""]

    remaining = len(todo_idx)
    print(f"\nRemaining rows to process: {remaining}")

    # Stop if all rows have been processed
    if remaining == 0:
        print("All done")
        break

    # Stop if the current run has reached the maximum allowed number of batches
    if batch_counter >= MAX_BATCHES:
        print(f"Reached MAX_BATCHES={MAX_BATCHES}. Stop here.")
        break

    # Select the next batch of rows to process
    batch_idx = todo_idx[:BATCH_SIZE]
    batch_counter += 1

    # Determine batch number based on completed rows
    completed = int((df["lyrics_status"] != "").sum())
    batch_no = (completed // BATCH_SIZE) + 1
    batch_file = os.path.join(OUT_DIR, f"batch{batch_no:02d}.csv")

    print(f"Processing batch{batch_no:02d} | rows {batch_idx[0]} ... {batch_idx[-1]}")

    # Process each row in the current batch
    for j, idx in enumerate(batch_idx, start=1):
        artist = str(df.loc[idx, "artist_name"]).strip()
        title  = str(df.loc[idx, "track_name"]).strip()

        url = ""
        try:
            # Build the AZLyrics URL and fetch lyrics
            url = build_azlyrics_url(artist, title)
            headers = make_headers()
            lyrics = fetch_azlyrics(url, headers=headers)

            # Save successful results
            df.at[idx, "azlyrics_url"] = url
            df.at[idx, "lyrics"] = lyrics
            df.at[idx, "lyrics_status"] = "ok"

        except Exception as e:
            # Save failure information for later review
            df.at[idx, "azlyrics_url"] = url
            df.at[idx, "lyrics"] = ""
            df.at[idx, "lyrics_status"] = f"fail: {type(e).__name__}"

        # Per request: random sleep of 4–8 seconds
        time.sleep(random.uniform(4, 9))

        # Every 10 requests: take an additional longer break
        if j % 10 == 0:
            print(f"Processed {j}/{len(batch_idx)} in this batch, taking a long break...")
            time.sleep(random.uniform(18, 29))

    # Save current batch results
    df.loc[batch_idx].to_csv(batch_file, index=False, encoding="utf-8-sig")
    print("Saved batch result to:", batch_file)

    # Save overall progress (for resume-safe execution)
    df.to_csv(PROGRESS_CSV, index=False, encoding="utf-8-sig")
    print("Saved progress to:", PROGRESS_CSV)

    # Extra long break between batches (safer)
    # time.sleep(random.uniform(45, 70))


In [1]:
# ------------------------------------------------------------
# Alternative scraper using Selenium (fallback solution)
# ------------------------------------------------------------
# In some cases AZLyrics may block direct HTTP requests and
# return a "request for access" page instead of the actual lyrics.
# When this happens, the requests-based scraper cannot locate
# the lyrics <div> in the HTML structure.
#
# To handle this situation, we implement an alternative method
# using Selenium with undetected_chromedriver. This approach
# launches a real browser instance and renders the page like a
# normal user, which can bypass simple anti-bot protections.
#
# The HTML source is then parsed using the same BeautifulSoup
# logic as in the original scraper.
# ------------------------------------------------------------

import re
from bs4 import BeautifulSoup
from requests.api import options
import undetected_chromedriver as uc
import time


def fetch_azlyrics_with_selenium(url: str) -> str:
    """
    Fetch lyrics text from an AZLyrics webpage using Selenium.

    This method loads the page through a real browser using
    undetected_chromedriver, allowing the script to retrieve the
    fully rendered HTML even when the site blocks automated requests.

    Input:
        url (str): URL of the AZLyrics page.

    Output:
        str: Cleaned lyrics text extracted from the page.
    """

    # ------------------------------------------------------------
    # Step 1: Configure Chrome options
    # ------------------------------------------------------------
    options = uc.ChromeOptions()

    # If you do not want the browser window to appear,
    # you can enable headless mode by uncommenting the line below.
    # options.add_argument('--headless')

    # Initialize Chrome driver
    # version_main ensures compatibility with the installed Chrome version
    driver = uc.Chrome(options=options, version_main=145)

    try:

        # ------------------------------------------------------------
        # Step 2: Load the target webpage
        # ------------------------------------------------------------
        driver.get(url)

        # Wait briefly to ensure the page is fully rendered
        # (important for pages protected by anti-bot checks)
        time.sleep(3)

        # ------------------------------------------------------------
        # Step 3: Retrieve rendered HTML
        # ------------------------------------------------------------
        page_source = driver.page_source

        # Parse the HTML using BeautifulSoup
        soup = BeautifulSoup(page_source, "html.parser")

        # ------------------------------------------------------------
        # Step 4: Locate the lyrics block
        # ------------------------------------------------------------
        main = soup.select_one("div.col-xs-12.col-lg-8.text-center")
        divs = main.find_all("div") if main else soup.find_all("div")

        lyrics = None

        # Identify the correct div containing the lyrics
        for d in divs:
            if d.get("class") is None and d.get("id") is None:
                text = d.get_text("\n", strip=True)

                # Heuristic rule: lyrics usually have many lines and characters
                if len(text.splitlines()) >= 5 and len(text) > 200:
                    lyrics = text
                    break

        # If no lyrics were found, raise an error
        if not lyrics:
            raise RuntimeError(
                "Failed to locate lyrics div. The site might still be blocking the request "
                "or the page structure has changed."
            )

        # ------------------------------------------------------------
        # Step 5: Clean excessive blank lines
        # ------------------------------------------------------------
        lyrics = re.sub(r"\n{3,}", "\n\n", lyrics).strip()

        return lyrics

    finally:
        # Always close the browser to release system resources
        driver.quit()


# ------------------------------------------------------------
# Test example
# ------------------------------------------------------------

url = "https://www.azlyrics.com/lyrics/taylorswift/blankspace.html"

try:
    lyrics = fetch_azlyrics_with_selenium(url)

    print("Successfully fetched lyrics! First 300 characters:\n")
    print(lyrics[:300])

except Exception as e:
    print(f"Scraping failed: {e}")

Successfully fetched lyrics! First 300 characters:

Nice to meet you, where you been?
I could show you incredible things
Magic, madness, heaven, sin
Saw you there, and I thought
"Oh my God, look at that face
You look like my next mistake
Love's a game, wanna play?"
Ayy
New money, suit and tie
I can read you like a magazine
Ain't it funny? Rumors fly

